In [ ]:
"""
EOS - YOLO Object Detection System
==================================
Real-time object detection with ESP32 camera stream and TTS feedback.

Author: Enhanced Version
"""

import cv2
import numpy as np
import requests
import pyttsx3
import threading
import queue
import time
import logging
from dataclasses import dataclass, field
from typing import Optional, List, Dict
from ultralytics import YOLO


# =========================================================
# CONFIGURATION
# =========================================================

@dataclass
class Config:
    """System configuration - غيّر الإعدادات هنا فقط"""
    
    # Camera settings
    camera_url: str = "http://10.97.160.127/cam-mid.jpg"
    
    # Model settings
    weights_path: str = "yolo26m.pt"
    confidence: float = 0.5
    
    # TTS settings
    cooldown_seconds: float = 5.0
    tts_rate: int = 150
    
    # Network settings
    request_timeout: int = 10
    
    # Display settings
    window_name: str = "EOS - YOLO Object Detection"
    flip_code: int = -1  # -1: flip both, 0: vertical, 1: horizontal, None: no flip
    
    # Performance settings
    frame_skip: int = 0  # Process every Nth frame (0 = all frames)
    max_queue_size: int = 10
    
    # Logging
    log_level: str = "INFO"


# =========================================================
# LOGGER SETUP
# =========================================================

def setup_logging(level: str = "INFO") -> logging.Logger:
    """Setup professional logging with colors and timestamps"""
    logging.basicConfig(
        level=getattr(logging, level.upper()),
        format='%(asctime)s [%(name)s] %(levelname)s: %(message)s',
        datefmt='%H:%M:%S'
    )
    return logging.getLogger("EOS")


# =========================================================
# TEXT-TO-SPEECH MANAGER
# =========================================================

class TTSManager:
    """
    Thread-safe Text-to-Speech manager with cooldown system.
    يدير النطق في Thread منفصل لعدم تعليق البرنامج
    """
    
    def __init__(self, config: Config):
        self.config = config
        self.logger = logging.getLogger("EOS.TTS")
        
        # Initialize TTS engine
        self.engine = pyttsx3.init()
        self.engine.setProperty("rate", config.tts_rate)
        
        # Thread-safe queue with max size
        self.speech_queue: queue.Queue = queue.Queue(maxsize=config.max_queue_size)
        self.last_spoken: Dict[str, float] = {}
        
        # Threading controls
        self._stop_event = threading.Event()
        self._thread = threading.Thread(target=self._worker, daemon=True, name="TTS-Worker")
        
    def start(self) -> None:
        """Start TTS worker thread"""
        self._thread.start()
        self.logger.info("TTS manager started")
        
    def stop(self) -> None:
        """Graceful shutdown of TTS"""
        self.logger.debug("Stopping TTS manager...")
        self._stop_event.set()
        
        # Signal worker to exit
        try:
            self.speech_queue.put_nowait(None)
        except queue.Full:
            pass
            
        self._thread.join(timeout=3)
        self.logger.info("TTS manager stopped")
        
    def _worker(self) -> None:
        """TTS worker thread - يعمل في الخلفية"""
        while not self._stop_event.is_set():
            try:
                text = self.speech_queue.get(timeout=0.5)
                if text is None:
                    break
                    
                try:
                    self.engine.say(text)
                    self.engine.runAndWait()
                except Exception as e:
                    self.logger.error(f"TTS engine error: {e}")
                    
                self.speech_queue.task_done()
                
            except queue.Empty:
                continue
                
    def enqueue(self, text: str) -> bool:
        """
        إضافة نص للنطق مع نظام Cooldown
        
        Args:
            text: النص المراد نطقه
            
        Returns:
            True إذا تمت الإضافة، False إذا تم التخطي (cooldown أو queue ممتلئة)
        """
        now = time.time()
        last = self.last_spoken.get(text, 0.0)
        
        # Cooldown check
        if now - last < self.config.cooldown_seconds:
            return False
            
        self.last_spoken[text] = now
        
        try:
            self.speech_queue.put_nowait(text)
            self.logger.info(f"Speaking: {text}")
            return True
        except queue.Full:
            self.logger.warning("Speech queue full, skipping TTS")
            return False
            
    def clear_queue(self) -> None:
        """مسح قائمة الانتظار"""
        cleared = 0
        while not self.speech_queue.empty():
            try:
                self.speech_queue.get_nowait()
                self.speech_queue.task_done()
                cleared += 1
            except queue.Empty:
                break
        if cleared > 0:
            self.logger.debug(f"Cleared {cleared} items from TTS queue")


# =========================================================
# FRAME FETCHER
# =========================================================

class FrameFetcher:
    """Handles fetching frames from ESP32 camera stream"""
    
    def __init__(self, config: Config):
        self.config = config
        self.logger = logging.getLogger("EOS.Camera")
        self.url = config.camera_url
        self.timeout = config.request_timeout
        
    def fetch(self) -> Optional[np.ndarray]:
        """
        جلب إطار من الكاميرا
        
        Returns:
            OpenCV frame (BGR) أو None في حالة الفشل
        """
        try:
            response = requests.get(self.url, timeout=self.timeout)
            response.raise_for_status()
            
            # Decode JPEG to OpenCV frame
            image_array = np.frombuffer(response.content, dtype=np.uint8)
            frame = cv2.imdecode(image_array, cv2.IMREAD_COLOR)
            
            if frame is None:
                self.logger.warning("Failed to decode JPEG frame")
                return None
                
            return frame
            
        except requests.exceptions.Timeout:
            self.logger.error(f"Camera request timeout ({self.timeout}s)")
        except requests.exceptions.ConnectionError:
            self.logger.error("Camera connection refused - check ESP32 IP")
        except requests.exceptions.RequestException as e:
            self.logger.error(f"Camera request failed: {e}")
        except Exception as e:
            self.logger.error(f"Unexpected frame fetch error: {e}")
            
        return None


# =========================================================
# DETECTION ENGINE
# =========================================================

class DetectionEngine:
    """YOLO object detection wrapper with result processing"""
    
    def __init__(self, config: Config):
        self.config = config
        self.logger = logging.getLogger("EOS.Detection")
        
        self.logger.info(f"Loading YOLO model: {config.weights_path}")
        self.model = YOLO(config.weights_path)
        self.logger.info("Model loaded successfully")
        
    def detect(self, frame: np.ndarray) -> tuple:
        """
        تشغيل الكشف على الإطار
        
        Args:
            frame: إطار OpenCV (BGR)
            
        Returns:
            (annotated_frame, labels_list)
        """
        results = self.model(frame, conf=self.config.confidence, verbose=False)[0]
        annotated = results.plot()
        
        labels = []
        if results.boxes is not None:
            for box in results.boxes:
                cls_id = int(box.cls[0])
                label = self.model.names.get(cls_id, f"class_{cls_id}")
                labels.append(label)
                
        return annotated, labels


# =========================================================
# DISPLAY MANAGER
# =========================================================

class DisplayManager:
    """Handles visualization, overlays, and UI interactions"""
    
    def __init__(self, config: Config):
        self.config = config
        self.logger = logging.getLogger("EOS.Display")
        self.window_name = config.window_name
        
    def setup(self) -> None:
        """Initialize display window"""
        cv2.namedWindow(self.window_name, cv2.WINDOW_NORMAL)
        self.logger.debug("Display window initialized")
        
    def add_overlay(self, frame: np.ndarray, labels: List[str], fps: float) -> np.ndarray:
        """
        إضافة معلومات على الإطار
        
        Args:
            frame: الإطار الأصلي
            labels: قائمة الأجسام المكتشفة
            fps: معدل الإطارات
            
        Returns:
            الإطار مع المعلومات المضافة
        """
        h, w = frame.shape[:2]
        
        # Detection labels (top-left)
        if labels:
            unique_text = ", ".join(dict.fromkeys(labels))
            # Background rectangle for better readability
            text_size = cv2.getTextSize(f"Detected: {unique_text}", cv2.FONT_HERSHEY_SIMPLEX, 0.7, 2)[0]
            cv2.rectangle(frame, (5, 5), (15 + text_size[0], 40), (0, 0, 0), -1)
            cv2.putText(
                frame,
                f"Detected: {unique_text}",
                (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (0, 255, 0),
                2,
            )
            
        # FPS counter (bottom-left)
        fps_color = (0, 255, 0) if fps >= 15 else (0, 165, 255) if fps >= 8 else (0, 0, 255)
        cv2.putText(
            frame,
            f"FPS: {fps:.1f}",
            (10, h - 15),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            fps_color,
            2,
        )
        
        # Status indicator (top-right)
        status = "● LIVE"
        cv2.putText(
            frame,
            status,
            (w - 100, 30),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (0, 0, 255),
            2,
        )
        
        return frame
        
    def show(self, frame: np.ndarray) -> bool:
        """
        عرض الإطار والتحقق من إشارة الإيقاف
        
        Returns:
            False إذا ضغط المستخدم 'q'
        """
        cv2.imshow(self.window_name, frame)
        return cv2.waitKey(1) & 0xFF != ord("q")
        
    def cleanup(self) -> None:
        """تنظيف موارد العرض"""
        cv2.destroyAllWindows()
        self.logger.debug("Display resources cleaned")


# =========================================================
# MAIN APPLICATION
# =========================================================

class EOSApp:
    """
    التطبيق الرئيسي - يدير جميع المكونات
    """
    
    def __init__(self, config: Optional[Config] = None):
        self.config = config or Config()
        self.logger = setup_logging(self.config.log_level)
        
        # Initialize components
        self.camera = FrameFetcher(self.config)
        self.detector = DetectionEngine(self.config)
        self.tts = TTSManager(self.config)
        self.display = DisplayManager(self.config)
        
        # State
        self.frame_count = 0
        self.running = False
        
    def start(self) -> None:
        """تشغيل التطبيق"""
        self.logger.info("=" * 60)
        self.logger.info("  EOS - YOLO Object Detection System")
        self.logger.info("=" * 60)
        self.logger.info(f"Camera URL: {self.config.camera_url}")
        self.logger.info(f"Model: {self.config.weights_path}")
        self.logger.info(f"Confidence: {self.config.confidence}")
        self.logger.info("Press 'q' to quit | Ctrl+C to force stop")
        self.logger.info("-" * 60)
        
        # Start components
        self.tts.start()
        self.display.setup()
        self.running = True
        
        try:
            self._main_loop()
        except KeyboardInterrupt:
            self.logger.info("Interrupted by user (Ctrl+C)")
        finally:
            self.shutdown()
            
    def _main_loop(self) -> None:
        """الحلقة الرئيسية للمعالجة"""
        last_time = time.time()
        fps = 0.0
        
        while self.running:
            # Fetch frame
            frame = self.camera.fetch()
            if frame is None:
                time.sleep(0.3)
                continue
                
            # Apply flip if configured
            if self.config.flip_code is not None:
                frame = cv2.flip(frame, self.config.flip_code)
                
            # Frame skip logic (performance optimization)
            self.frame_count += 1
            if self.config.frame_skip > 0:
                if self.frame_count % (self.config.frame_skip + 1) != 0:
                    continue
                    
            # Run detection
            annotated, labels = self.detector.detect(frame)
            
            # Calculate FPS
            current_time = time.time()
            delta = current_time - last_time
            fps = 1.0 / delta if delta > 0 else 0.0
            last_time = current_time
            
            # Add UI overlay
            annotated = self.display.add_overlay(annotated, labels, fps)
            
            # Log and speak detections
            if labels:
                unique_labels = list(dict.fromkeys(labels))
                detection_text = ", ".join(unique_labels)
                self.logger.info(f"Detected: {detection_text}")
                self.tts.enqueue(detection_text)
                
            # Display frame
            if not self.display.show(annotated):
                self.logger.info("Quit signal received ('q' pressed)")
                self.running = False
                
    def shutdown(self) -> None:
        """إيقاف آمن ومنظم"""
        self.logger.info("Initiating shutdown sequence...")
        self.running = False
        
        # Stop components in order
        self.tts.stop()
        self.display.cleanup()
        
        self.logger.info("=" * 60)
        self.logger.info("EOS stopped successfully.")
        self.logger.info("=" * 60)


# =========================================================
# ENTRY POINT
# =========================================================

if __name__ == "__main__":
    # إعداداتك - عدل هنا حسب احتياجك
    config = Config(
        camera_url="http://10.97.160.127/cam-mid.jpg",
        weights_path="yolo26m.pt",
        confidence=0.5,
        cooldown_seconds=5.0,
        tts_rate=150,
        request_timeout=10,
        window_name="EOS - YOLO Object Detection",
        flip_code=-1,           # اقلب الصورة (both axes)
        frame_skip=0,           # 0 = عالج كل الإطارات
        max_queue_size=10,
        log_level="INFO"
    )
    
    app = EOSApp(config)
    app.start()


0: 480x640 (no detections), 2458.8ms
Speed: 356.8ms preprocess, 2458.8ms inference, 188.7ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 3 persons, 214.1ms
Speed: 3.7ms preprocess, 214.1ms inference, 516.5ms postprocess per image at shape (1, 3, 480, 640)
Detected: person
Detected: person
Detected: person

0: 480x640 2 persons, 238.7ms
Speed: 5.4ms preprocess, 238.7ms inference, 2.1ms postprocess per image at shape (1, 3, 480, 640)
Detected: person
Detected: person

0: 480x640 2 persons, 230.6ms
Speed: 7.2ms preprocess, 230.6ms inference, 2.0ms postprocess per image at shape (1, 3, 480, 640)
Detected: person
Detected: person

0: 480x640 1 person, 224.1ms
Speed: 3.7ms preprocess, 224.1ms inference, 2.1ms postprocess per image at shape (1, 3, 480, 640)
Detected: person

0: 480x640 1 person, 200.3ms
Speed: 3.5ms preprocess, 200.3ms inference, 2.0ms postprocess per image at shape (1, 3, 480, 640)
Detected: person

0: 480x640 1 person, 209.0ms
Speed: 7.2ms preprocess, 209.0m